# 5 Advanced Visualizations

### 📊 What is the visualization about?
We want to create a shot chart for a given NHL team, showing the location of its shots on the ice to visualize shooting patterns and identify areas of offensive strength.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from ift6758.features.simple_visualizations import games_list
import plotly.graph_objects as go

- `number_games()` calculates the number of games played by each team during a given season.

In [2]:
def number_games(season) -> dict:
    # Get all the games dataframe
    df_games = games_list(season, events=False)

    # Keep regular season and playoffs only (game type code in the game id)
    df_games = df_games[df_games["game_type"].isin([2, 3])]

    # Calculate the number of game for each team
    number_games_per_team = {}
    for game in df_games.itertuples():
        homeTeam_id = getattr(game, "homeTeam_name")
        awayTeam_id = getattr(game, "awayTeam_name")
        number_games_per_team[homeTeam_id] = number_games_per_team.get(homeTeam_id, 0) + 1
        number_games_per_team[awayTeam_id] = number_games_per_team.get(awayTeam_id, 0) + 1

    return number_games_per_team

- `shot_data` contains all the information needed to generate the desired visualization. It returns a DataFrame containing, for each team, its name, the number of games played during the season, and the coordinates of every shot and goal recorded throughout the season.

In [3]:

def shot_data(season):
    """
    Gets the data from json (possibly by calling the distance function) for the season and extracts the shot coordinates to a more usable format
    :param season: 4 digit year of the season start
    :return: a dataframe containing each team's name, number of games, and an array of x,y shot coordinates (1 line per shot, columns x and y)
    """
    # Get each team's number of games
    team_game_number = number_games(season)

    # Link each team_id to its abbrev
    df_games = games_list(season, events=False)
    id_to_abbrev = {}
    for game in df_games.itertuples():
        id_to_abbrev[getattr(game, "homeTeam_id")] = getattr(game, "homeTeam_name")
        id_to_abbrev[getattr(game, "awayTeam_id")] = getattr(game, "awayTeam_name")

    # Get the shots and goals dataframe for the given season
    df_events = games_list(season)

    # Keep regular season and playoffs
    df_shots = df_events[
        df_events["game_type"].isin(["regular season", "playoffs"])
    ].copy()

    # Put all the shots on the same side (use same logic as distance() from simple_visualization.py)
    home_shoots = df_shots["team_id"] == df_shots["homeTeam_id"]
    attacks_left = np.where(
        home_shoots,
        df_shots["homeTeamDefendingSide"] == "right",
        df_shots["homeTeamDefendingSide"] == "left"
    )

    # Flip the shots by 180°­­­­ if they were taken from the left side of the pitch
    df_shots.loc[attacks_left, ["x_coord", "y_coord"]] *= -1

    rows = []
    for team_id, df_team in df_shots.groupby("team_id"):
        team = id_to_abbrev[team_id]
        rows.append({
            "team": team,
            "games": team_game_number[team],
            "shots": df_team[["x_coord", "y_coord"]].T
        })

    return pd.DataFrame(rows)

print(shot_data(2022).head())

                  team  games  \
0    New Jersey Devils     94   
1   New York Islanders     88   
2     New York Rangers     89   
3  Philadelphia Flyers     82   
4  Pittsburgh Penguins     82   

                                               shots  
0           725    727    731    735    736    73...  
1           859    860    861    862    865    86...  
2           115    116    118    122    123    12...  
3           724    726    728    729    730    73...  
4           643    644    646    647    649    65...  


In [4]:

def kde():
    """
    Calculates kde when given an array of shot coordinates
    :return: shot density
    """



In [5]:

class ShotMap:

    def __init__(self, season):
        """

        :param season: 4 digit year of the season start
        """
        self.season = season
        self.shots = shot_data(season)
        self.kdes = []

    def calculate_kdes(self):
        """
        This calculates league average shot rate per hour (league_sph) and excess shot rate per hour for each team, in the form of shot density using kde. Also saves raw difference in shots per team hour
        :return:
        """
        # first, calculate kde for league average and save the factor, which will be given to the team kdes

        # calculate shot rate per hour density for each team using kde:
        for team_coords in self.shots["shots"]:
            print(team_coords)
            self.kdes.append(stats.gaussian_kde(team_coords))
        # densities will need to be weighed by shot rate

        # convert teams' sph to excess sph using league average
        # calculate raw difference in shots per team hour


In [8]:
shots2020 = ShotMap(2022)


In [9]:
print(shots2020.shots.shape)
print(shots2020.shots.columns)


(32, 3)
Index(['team', 'games', 'shots'], dtype='object')


In [10]:
shots2020.calculate_kdes()
shots2020.shots.head()
print(shots2020.kdes)

         725    727    731    735    736    737    738    739    740    741    \
x_coord   75.0   62.0   80.0  -92.0   80.0   61.0   83.0   82.0   80.0   33.0   
y_coord   11.0   24.0  -18.0  -27.0   -5.0  -10.0   -4.0   -5.0   -1.0  -16.0   

         ...  86307  86308  86315  86316  86317  86319  86324  86329  86330  \
x_coord  ...   31.0   38.0   77.0   11.0   77.0  -41.0   87.0   86.0   30.0   
y_coord  ...  -35.0  -13.0  -18.0   -3.0   -1.0    0.0   31.0    9.0   24.0   

         86331  
x_coord   37.0  
y_coord   20.0  

[2 rows x 3177 columns]
         859    860    861    862    865    866    867    870    871    877    \
x_coord   74.0  -90.0   69.0   40.0   57.0   86.0   86.0   72.0   77.0   87.0   
y_coord    1.0  -21.0   24.0  -24.0  -12.0   30.0    5.0   37.0   14.0   -6.0   

         ...  83746  83747  83748  83749  83752  83758  83763  83766  83770  \
x_coord  ...   59.0    2.0   80.0   38.0   67.0   77.0  -69.0  -83.0   47.0   
y_coord  ...  -19.0  -40.0   -2.0   25.0

In [11]:
print(shots2020.kdes[0])
X, Y = np.mgrid[-100:100:200j, -100:100:200j]
positions = np.vstack([X.ravel(), Y.ravel()])
# différence entre deux équipes:
plt.imshow(np.reshape(shots2020.kdes[0](positions).T, X.shape) - np.reshape(shots2020.kdes[1](positions).T, X.shape), cmap = "gnuplot2")
plt.show()

<Figure size 640x480 with 1 Axes>

In [ ]:
def interactive_shot_map(shot_map):
    """
    :param shot_map: ShotMap object
    :return: Creates html interactive figure ?
    """
    # checks if shot_map has kdes, if not, calls shot_map.calculate_kde()
    if len(shot_map.kdes) == 0:
        shot_map.calculate_kdes()

    df = shot_map.shots
    # For now not the right data
    coords = df["shots"][0] #For initialisation of the figure
    # coords = shot_map.kdes[0]
    team_list = df["team"].drop_duplicates()

    teams = []
    for i in range(len(team_list)):
        it = dict(label = team_list[i], args=[{"z": [df["shots"][i].values.tolist()]}], method="restyle")
        teams.append(it)

    # CODE TAKEN FROM PLOTLY DEMO
    # create figure
    fig = go.Figure()

    # Add surface trace
    fig.add_trace(go.Heatmap(z=coords.values.tolist(), colorscale="Viridis")) #Changed code

    # Update plot sizing
    fig.update_layout(
        width=800,
        height=900,
        autosize=False,
        margin=dict(t=0, b=0, l=0, r=0),
        template="plotly_white",
    )

    # Update 3D scene options
    fig.update_scenes(
        aspectratio=dict(x=1, y=1, z=0.7),
        aspectmode="manual"
    )

    # Add dropdown
    fig.update_layout(
        updatemenus=[
            dict(
                buttons = teams, #Changed code here
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
        ]
    )

    # Add annotation
    fig.update_layout(
        annotations=[
            dict(text="Team:", showarrow=False,
            x=0, y=1.085, yref="paper", align="left")
        ]
    )

    # From demo in blog template
    # fig.write_html("../../figures/interactive_figure.html", include_plotlyjs="cdn", full_html=False)
    fig.show()

In [14]:
interactive_shot_map(shots2020)